This notebook generates final-report plots only. It does not rerun models, selectors, LLM calls, or CLIP.

It reads existing result artifacts and prediction artifacts, saves only professor-facing plots under `results/analysis_plots/`, and records created/skipped plots in `plot_manifest.csv`.


## Setup and Data Discovery


In [ ]:

from pathlib import Path
import math
import warnings

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / 'results' / 'analysis_plots'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = OUTPUT_DIR / 'plot_manifest.csv'
DPI = 300

METHOD_LABELS = {
    'llm': 'LLM',
    'mrmr': 'mRMR',
    'stable_core_llm_fill': 'Stable Core + LLM Fill',
    'llm_then_mrmr': 'LLM -> mRMR',
    'llm_then_boruta': 'LLM -> Boruta',
    'domain_rule_baseline': 'Domain Rules',
    'pca': 'PCA',
    'boruta': 'Boruta',
}
DATASET_LABELS = {
    'homecredit': 'Home Credit',
    'lendingclub_v2': 'LendingClub v2',
    'lendingclub': 'LendingClub v1',
}
KEY_SELECTORS = ['llm', 'mrmr', 'stable_core_llm_fill', 'llm_then_mrmr']

PREFERRED_SOURCES = {
    'cross_final': 'results/cross_dataset_v2/analysis/final.csv',
    'best_overall': 'results/cross_dataset_v2/analysis/best_overall.csv',
    'best_llm_family': 'results/cross_dataset_v2/analysis/best_llm_family.csv',
    'deltas_vs_mrmr': 'results/cross_dataset_v2/analysis/deltas_vs_mrmr.csv',
    'family_summary': 'results/cross_dataset_v2/analysis/family_summary.csv',
    'semantic_summary': 'results/cross_dataset_v2/analysis/semantic_summary.csv',
    'top_feature_evidence': 'results/cross_dataset_v2/analysis/top_feature_evidence.csv',
    'homecredit_final': 'results/homecredit/final_comparison_table.csv',
    'lendingclub_v2_final': 'results/lendingclub_v2/final_comparison_table.csv',
    'lendingclub_v1_final': 'results/lendingclub/final_comparison_table.csv',
    'homecredit_semantic': 'results/homecredit/semantic_coverage_table.csv',
    'lendingclub_v2_semantic': 'results/lendingclub_v2/semantic_coverage_table.csv',
    'lendingclub_v1_semantic': 'results/lendingclub/semantic_coverage_table.csv',
    'lc_v1_gap_audit': 'reports/lendingclub_v1_feature_engineering_gap_audit.md',
    'lc_v2_inventory': 'reports/lendingclub_v2_feature_inventory.md',
    'lc_v2_metadata_quality': 'reports/lendingclub_v2_metadata_quality_audit.md',
    'lc_v2_pre_matrix_approval': 'reports/lendingclub_v2_final_pre_matrix_approval.md',
}

manifest_rows = []
source_paths = {}

def repo_path(path_text):
    return PROJECT_ROOT / path_text

def discover_path(path_text):
    preferred = repo_path(path_text)
    if preferred.exists():
        return preferred

    parts = Path(path_text).parts
    name = Path(path_text).name

    # Generic filenames such as final_comparison_table.csv must not cross dataset boundaries.
    if len(parts) >= 3 and parts[0] == 'results':
        dataset = parts[1]
        for root in [PROJECT_ROOT / 'results' / dataset, PROJECT_ROOT / 'results_v1' / dataset]:
            if root.exists():
                matches = sorted(root.rglob(name))
                if matches:
                    return matches[0]
        return preferred

    for root_name in ['reports', 'data', 'results', 'results_v1']:
        root = PROJECT_ROOT / root_name
        if not root.exists():
            continue
        matches = sorted(root.rglob(name))
        if matches:
            return matches[0]
    return preferred

for key, rel in PREFERRED_SOURCES.items():
    source_paths[key] = discover_path(rel)

missing_sources = [f'{key}: {path}' for key, path in source_paths.items() if not path.exists()]
if missing_sources:
    print('Missing preferred sources:')
    for item in missing_sources:
        print('  -', item)


def read_csv_source(key):
    path = source_paths[key]
    if not path.exists():
        warnings.warn(f'Missing source file for {key}: {path}')
        return pd.DataFrame()
    return pd.read_csv(path)


def rel(path):
    try:
        return str(Path(path).resolve().relative_to(PROJECT_ROOT.resolve())).replace('\\', '/')
    except Exception:
        return str(path).replace('\\', '/')


def method_label(value):
    return METHOD_LABELS.get(str(value), str(value))


def dataset_label(value):
    return DATASET_LABELS.get(str(value), str(value))


def model_label(value):
    return 'LR' if str(value).lower() == 'lr' else 'CatBoost' if str(value).lower() == 'catboost' else str(value)


def family_label(value):
    return {'llm_family': 'LLM family', 'statistical': 'Statistical'}.get(str(value), str(value))


def add_manifest(plot_file, plot_title, data_source_files, status, reason_if_skipped, notes_for_report):
    manifest_rows.append({
        'plot_file': plot_file,
        'plot_title': plot_title,
        'data_source_files': '; '.join(rel(p) for p in data_source_files),
        'status': status,
        'reason_if_skipped': reason_if_skipped,
        'notes_for_report': notes_for_report,
    })


def save_fig(fig, filename):
    path = OUTPUT_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=DPI, bbox_inches='tight')
    plt.close(fig)
    return path


def style_axes(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.25, linewidth=0.8)


def require_columns(df, columns):
    return [col for col in columns if col not in df.columns]

cross_final = read_csv_source('cross_final')
best_overall = read_csv_source('best_overall')
family_summary = read_csv_source('family_summary')
semantic_summary = read_csv_source('semantic_summary')
homecredit_final = read_csv_source('homecredit_final')
lendingclub_v2_final = read_csv_source('lendingclub_v2_final')
lendingclub_v1_final = read_csv_source('lendingclub_v1_final')

print('Data sources discovered:')
for key, path in source_paths.items():
    print(f"{key}: {'FOUND' if path.exists() else 'MISSING'} - {rel(path)}")


## Core Comparison Plots


In [ ]:

# Plot 1: LendingClub v1 vs v2 metadata readiness
plot_file = '01_lendingclub_v1_vs_v2_metadata_readiness.png'
title = 'LendingClub v1 vs v2 metadata readiness'
caption = 'LendingClub v2 removes the main v1 weakness: incomplete metadata coverage for LLM screening.'
try:
    readiness = pd.DataFrame({
        'dataset': ['LendingClub v1', 'LendingClub v2'],
        'candidate_features': [300, 675],
        'usable_descriptions': [76, 675],
        'description_coverage_pct': [76 / 300 * 100, 100.0],
    })
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    x = range(len(readiness))
    width = 0.25
    ax.bar([i - width for i in x], readiness['candidate_features'], width=width, label='Candidate features', color='#4C78A8')
    ax.bar(x, readiness['usable_descriptions'], width=width, label='Usable descriptions', color='#59A14F')
    ax2 = ax.twinx()
    ax2.bar([i + width for i in x], readiness['description_coverage_pct'], width=width, label='Description coverage %', color='#F28E2B', alpha=0.85)
    ax.set_xticks(list(x))
    ax.set_xticklabels(readiness['dataset'])
    ax.set_ylabel('Feature count')
    ax2.set_ylabel('Description coverage')
    ax2.yaxis.set_major_formatter(PercentFormatter())
    ax.set_title(title)
    ax.set_ylim(0, max(readiness['candidate_features']) * 1.18)
    ax2.set_ylim(0, 115)
    handles1, labels1 = ax.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(handles1 + handles2, labels1 + labels2, loc='upper left', frameon=False)
    ax.grid(axis='y', alpha=0.25)
    ax.spines['top'].set_visible(False)
    ax2.spines['top'].set_visible(False)
    save_fig(fig, plot_file)
    add_manifest(plot_file, title, [source_paths['lc_v1_gap_audit'], source_paths['lc_v2_inventory'], source_paths['lc_v2_metadata_quality'], source_paths['lc_v2_pre_matrix_approval']], 'created', '', caption)
except Exception as exc:
    add_manifest(plot_file, title, [source_paths['lc_v1_gap_audit'], source_paths['lc_v2_pre_matrix_approval']], 'skipped', str(exc), caption)

# Plot 2: Top OOT AUC methods across Home Credit and LendingClub v2
plot_file = '02_top_oot_auc_best_methods.png'
title = 'Top OOT AUC methods across Home Credit and LendingClub v2'
caption = 'The best OOT pipelines in both datasets are LLM-family methods, but the winning variant differs by dataset.'
try:
    needed = ['dataset_name', 'model', 'selector', 'oot_auc']
    missing = require_columns(best_overall, needed)
    if best_overall.empty or missing:
        raise ValueError(f'best_overall missing required columns: {missing}')
    desired = best_overall[best_overall['dataset_name'].isin(['homecredit', 'lendingclub_v2'])].copy()
    desired = desired.sort_values(['dataset_name', 'model']).copy()
    desired['label'] = desired.apply(lambda r: f"{dataset_label(r['dataset_name'])} {model_label(r['model'])}: {method_label(r['selector'])}", axis=1)
    desired = desired.sort_values('oot_auc', ascending=True)
    if len(desired) != 4:
        raise ValueError(f'Expected four best-method rows, found {len(desired)}')
    fig, ax = plt.subplots(figsize=(9, 4.8))
    colors = ['#4C78A8' if 'Home Credit' in v else '#F28E2B' for v in desired['label']]
    ax.barh(desired['label'], desired['oot_auc'], color=colors)
    for y, value in enumerate(desired['oot_auc']):
        ax.text(value + 0.002, y, f'{value:.4f}', va='center', fontsize=9)
    ax.set_xlabel('OOT AUC')
    ax.set_title(title)
    ax.set_xlim(max(0.60, desired['oot_auc'].min() - 0.04), desired['oot_auc'].max() + 0.03)
    style_axes(ax)
    save_fig(fig, plot_file)
    add_manifest(plot_file, title, [source_paths['best_overall']], 'created', '', caption)
except Exception as exc:
    add_manifest(plot_file, title, [source_paths['best_overall']], 'skipped', str(exc), caption)

# Plot 3: LLM-family vs statistical family average OOT AUC
plot_file = '03_family_level_oot_auc.png'
title = 'LLM-family vs statistical family average OOT AUC'
caption = 'LLM-family methods are competitive or leading at family level, especially on LendingClub v2.'
try:
    needed = ['dataset_name', 'model', 'selector_family', 'oot_auc']
    missing = require_columns(family_summary, needed)
    if family_summary.empty or missing:
        raise ValueError(f'family_summary missing required columns: {missing}')
    df = family_summary[family_summary['selector_family'].isin(['llm_family', 'statistical'])].copy()
    df['row_label'] = df.apply(lambda r: f"{dataset_label(r['dataset_name'])} {model_label(r['model'])}", axis=1)
    pivot = df.pivot_table(index='row_label', columns='selector_family', values='oot_auc', aggfunc='mean')
    pivot = pivot.reindex(columns=['llm_family', 'statistical']).dropna(how='all')
    if pivot.empty:
        raise ValueError('No LLM-family/statistical rows available')
    fig, ax = plt.subplots(figsize=(9.5, 5.2))
    x = list(range(len(pivot)))
    width = 0.34
    ax.bar([i - width/2 for i in x], pivot['llm_family'], width=width, label='LLM family', color='#4C78A8')
    ax.bar([i + width/2 for i in x], pivot['statistical'], width=width, label='Statistical', color='#9C755F')
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=20, ha='right')
    ax.set_ylabel('Average OOT AUC')
    ax.set_title(title)
    ax.legend(frameon=False)
    style_axes(ax)
    ax.set_ylim(max(0.55, float(pivot.min().min()) - 0.04), min(0.82, float(pivot.max().max()) + 0.04))
    save_fig(fig, plot_file)
    add_manifest(plot_file, title, [source_paths['family_summary']], 'created', '', caption)
except Exception as exc:
    add_manifest(plot_file, title, [source_paths['family_summary']], 'skipped', str(exc), caption)

# Plot 4: Score PSI by key methods with PSI quality ranges
plot_file = '04_score_psi_key_methods.png'
title = 'Score PSI by key LLM-family and mRMR methods'
caption = 'Score PSI remains in the low range for key LLM-family and mRMR comparators.'
try:
    source = cross_final.copy()
    needed = ['dataset_name', 'model', 'selector', 'model_score_psi']
    missing = require_columns(source, needed)
    if source.empty or missing:
        raise ValueError(f'cross final missing required columns: {missing}')
    wanted = [
        ('homecredit', 'catboost', 'stable_core_llm_fill'),
        ('homecredit', 'catboost', 'mrmr'),
        ('lendingclub_v2', 'catboost', 'llm'),
        ('lendingclub_v2', 'catboost', 'mrmr'),
        ('homecredit', 'lr', 'stable_core_llm_fill'),
        ('homecredit', 'lr', 'mrmr'),
        ('lendingclub_v2', 'lr', 'llm'),
        ('lendingclub_v2', 'lr', 'mrmr'),
    ]
    rows = []
    for dataset_name, model, selector in wanted:
        match = source[(source['dataset_name'] == dataset_name) & (source['model'] == model) & (source['selector'] == selector)]
        if not match.empty:
            rows.append(match.iloc[0])
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError('No key method rows found for score PSI')
    df['label'] = df.apply(lambda r: f"{dataset_label(r['dataset_name'])} {model_label(r['model'])} {method_label(r['selector'])}", axis=1)
    fig, ax = plt.subplots(figsize=(10.5, 5.4))
    ax.axhspan(0, 0.10, color='#DDF2D1', alpha=0.9, label='Low / good (<0.10)')
    ax.axhspan(0.10, 0.25, color='#FFF2CC', alpha=0.8, label='Moderate / watch (0.10-0.25)')
    ax.axhspan(0.25, max(0.30, float(df['model_score_psi'].max()) + 0.03), color='#FADBD8', alpha=0.65, label='High / bad (>=0.25)')
    colors = ['#4C78A8' if str(sel) != 'mrmr' else '#9C755F' for sel in df['selector']]
    ax.bar(range(len(df)), df['model_score_psi'], color=colors, edgecolor='white')
    ax.set_xticks(range(len(df)))
    ax.set_xticklabels(df['label'], rotation=35, ha='right')
    ax.set_ylabel('Model score PSI')
    ax.set_title(title)
    ax.set_ylim(0, max(0.30, float(df['model_score_psi'].max()) + 0.03))
    ax.legend(frameon=False, loc='upper right')
    style_axes(ax)
    save_fig(fig, plot_file)
    add_manifest(plot_file, title, [source_paths['cross_final']], 'created', '', caption)
except Exception as exc:
    add_manifest(plot_file, title, [source_paths['cross_final']], 'skipped', str(exc), caption)


## Optional Prediction-Level and Semantic/Stability Plots


In [ ]:

# Helpers for prediction-level optional plots
TRUE_COLS = ['TARGET', 'y_true', 'target', 'label', 'default_flag', 'bad_flag']
SCORE_COLS = ['y_score', 'score', 'probability', 'pred_proba', 'y_pred_proba', 'oot_score', 'prediction']
TIME_COLS = ['issue_d', 'month', 'date', 'period', 'recent_decision', 'time_bucket']


def find_prediction_file(row):
    candidates = []
    row_data = dict(row)

    if 'output_folder' not in row_data or not isinstance(row_data.get('output_folder'), str):
        if not cross_final.empty and {'dataset_name', 'model', 'selector', 'output_folder'}.issubset(cross_final.columns):
            match = cross_final[
                (cross_final['dataset_name'] == row_data.get('dataset_name'))
                & (cross_final['model'] == row_data.get('model'))
                & (cross_final['selector'] == row_data.get('selector'))
            ]
            if not match.empty:
                row_data.update(match.iloc[0].to_dict())

    output_folder = row_data.get('output_folder')
    if isinstance(output_folder, str) and output_folder.strip():
        folder = Path(output_folder)
        if not folder.is_absolute():
            folder = PROJECT_ROOT / folder
        if folder.exists():
            candidates.extend(folder.rglob('oot_predictions.csv'))
            candidates.extend(folder.rglob('*prediction*.csv'))
            candidates.extend(folder.rglob('*score*.csv'))

    run_id = str(row_data.get('run_id', ''))
    if run_id:
        for root in [PROJECT_ROOT / 'results' / str(row_data.get('dataset_name', '')), PROJECT_ROOT / 'results']:
            if root.exists():
                candidates.extend(root.rglob(f'*{run_id}*'))

    seen = []
    for path in candidates:
        if path.is_file() and path.suffix.lower() in ['.csv', '.parquet'] and path not in seen:
            seen.append(path)
    for path in seen:
        try:
            sample = pd.read_csv(path, nrows=10) if path.suffix.lower() == '.csv' else pd.read_parquet(path).head(10)
        except Exception:
            continue
        true_col = next((col for col in TRUE_COLS if col in sample.columns), None)
        score_col = next((col for col in SCORE_COLS if col in sample.columns), None)
        if true_col and score_col:
            return path, true_col, score_col
    return None, None, None


def roc_curve_from_scores(y_true, y_score):
    df = pd.DataFrame({'y_true': y_true, 'y_score': y_score}).dropna()
    df['y_true'] = pd.to_numeric(df['y_true'], errors='coerce')
    df['y_score'] = pd.to_numeric(df['y_score'], errors='coerce')
    df = df.dropna().sort_values('y_score', ascending=False)
    positives = float((df['y_true'] == 1).sum())
    negatives = float((df['y_true'] == 0).sum())
    if positives == 0 or negatives == 0:
        raise ValueError('ROC requires both positive and negative labels')
    tps = (df['y_true'] == 1).cumsum()
    fps = (df['y_true'] == 0).cumsum()
    tpr = pd.concat([pd.Series([0.0]), tps / positives, pd.Series([1.0])], ignore_index=True)
    fpr = pd.concat([pd.Series([0.0]), fps / negatives, pd.Series([1.0])], ignore_index=True)
    auc = float(((fpr.diff().fillna(0)) * (tpr + tpr.shift(fill_value=0)) / 2).sum())
    return fpr, tpr, auc

# Plot 5: ROC curves for top four OOT methods if prediction files exist
plot_file = '05_oot_roc_curves_top4.png'
title = 'OOT ROC curves for top four methods'
caption = 'ROC curves are computed from saved OOT prediction-level scores only.'
try:
    needed = ['dataset_name', 'model', 'selector', 'oot_auc']
    missing = require_columns(best_overall, needed)
    if best_overall.empty or missing:
        raise ValueError(f'best_overall missing required columns: {missing}')
    top_rows = best_overall[best_overall['dataset_name'].isin(['homecredit', 'lendingclub_v2'])].copy()
    if len(top_rows) != 4:
        raise ValueError(f'Expected four top rows, found {len(top_rows)}')
    fig, ax = plt.subplots(figsize=(7, 6))
    used_files = []
    for _, row in top_rows.iterrows():
        pred_path, true_col, score_col = find_prediction_file(row)
        if pred_path is None:
            raise ValueError('Prediction-level OOT scores with y_true and y_score were not found; ROC curves cannot be reconstructed from aggregate AUC only.')
        pred = pd.read_csv(pred_path) if pred_path.suffix.lower() == '.csv' else pd.read_parquet(pred_path)
        fpr, tpr, auc = roc_curve_from_scores(pred[true_col], pred[score_col])
        label = f"{dataset_label(row['dataset_name'])} {model_label(row['model'])} {method_label(row['selector'])} (AUC {auc:.4f})"
        ax.plot(fpr, tpr, linewidth=2, label=label)
        used_files.append(pred_path)
    ax.plot([0, 1], [0, 1], color='#777777', linestyle='--', linewidth=1)
    ax.set_xlabel('False positive rate')
    ax.set_ylabel('True positive rate')
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8, loc='lower right')
    ax.grid(alpha=0.25)
    save_fig(fig, plot_file)
    add_manifest(plot_file, title, used_files, 'created', '', caption)
except Exception as exc:
    add_manifest(plot_file, title, [source_paths['best_overall']], 'skipped', 'Prediction-level OOT scores with y_true and y_score were not found; ROC curves cannot be reconstructed from aggregate AUC only.' if 'Prediction-level OOT scores' in str(exc) else str(exc), caption)

# Plot 6: Score PSI over time if period-level score distributions exist
plot_file = '06_score_psi_over_time_top4.png'
title = 'Score PSI over time for top methods'
caption = 'Score PSI over time is only valid when period-level score distributions and a DEV reference are available.'
try:
    valid_period_files = []
    for _, row in best_overall.iterrows():
        pred_path, true_col, score_col = find_prediction_file(row)
        if pred_path is None:
            continue
        sample = pd.read_csv(pred_path, nrows=20) if pred_path.suffix.lower() == '.csv' else pd.read_parquet(pred_path).head(20)
        time_col = next((col for col in TIME_COLS if col in sample.columns), None)
        split_col = next((col for col in ['split', 'split_segment', 'segment'] if col in sample.columns), None)
        if time_col and split_col:
            valid_period_files.append(pred_path)
    if not valid_period_files:
        raise ValueError('Period-level score distributions were not found; only aggregate DEV-to-OOT model_score_psi is available.')
    raise ValueError('Period-level score PSI plotting is not implemented for the discovered files because no validated DEV reference distribution was found.')
except Exception as exc:
    reason = 'Period-level score distributions were not found; only aggregate DEV-to-OOT model_score_psi is available.'
    add_manifest(plot_file, title, [source_paths['best_overall']], 'skipped', reason, caption)

# Plot 7: Semantic coverage of key selectors
plot_file = '07_semantic_coverage_key_selectors.png'
title = 'Semantic coverage of key selectors'
caption = 'LLM-family selectors preserve broad business-concept coverage; semantic coverage is a secondary advantage, not a dominance claim.'
try:
    needed = ['dataset_name', 'model', 'selector', 'semantic_group_count']
    missing = require_columns(semantic_summary, needed)
    if semantic_summary.empty or missing:
        raise ValueError(f'semantic_summary missing required columns: {missing}')
    df = semantic_summary[semantic_summary['selector'].isin(KEY_SELECTORS)].copy()
    df = df[df['dataset_name'].isin(['homecredit', 'lendingclub_v2'])]
    if df.empty:
        raise ValueError('No key selector semantic summary rows found')
    # Keep CatBoost only to avoid overcrowding, per request allowance.
    df = df[df['model'] == 'catboost'].copy()
    df['label'] = df.apply(lambda r: f"{dataset_label(r['dataset_name'])} {method_label(r['selector'])}", axis=1)
    df = df.sort_values(['dataset_name', 'semantic_group_count'])
    fig, ax = plt.subplots(figsize=(9.5, 5.2))
    colors = ['#4C78A8' if str(sel) != 'mrmr' else '#9C755F' for sel in df['selector']]
    ax.barh(df['label'], df['semantic_group_count'], color=colors)
    for y, value in enumerate(df['semantic_group_count']):
        ax.text(value + 0.1, y, f'{int(value)}', va='center', fontsize=9)
    ax.set_xlabel('Number of semantic groups')
    ax.set_title(title + ' (CatBoost)')
    style_axes(ax)
    save_fig(fig, plot_file)
    add_manifest(plot_file, title, [source_paths['semantic_summary']], 'created', '', caption)
except Exception as exc:
    add_manifest(plot_file, title, [source_paths['semantic_summary']], 'skipped', str(exc), caption)

# Plot 8: Stability vs OOT AUC tradeoff
plot_file = '08_stability_vs_oot_auc_tradeoff.png'
title = 'Stability vs OOT AUC tradeoff'
caption = 'LLM-family methods are competitive in OOT AUC, while mRMR remains the exact-stability reference.'
try:
    needed = ['dataset_name', 'model', 'selector', 'selector_family', 'oot_auc', 'nogueira_stability']
    missing = require_columns(cross_final, needed)
    if cross_final.empty or missing:
        raise ValueError(f'cross final missing required columns: {missing}')
    df = cross_final[cross_final['dataset_name'].isin(['homecredit', 'lendingclub_v2']) & cross_final['selector'].isin(KEY_SELECTORS)].copy()
    if df.empty:
        raise ValueError('No key selector stability rows found')
    fig, ax = plt.subplots(figsize=(8.8, 6.2))
    colors = {'llm_family': '#4C78A8', 'statistical': '#9C755F'}
    markers = {'homecredit': 'o', 'lendingclub_v2': 's'}
    for _, row in df.iterrows():
        ax.scatter(
            row['nogueira_stability'],
            row['oot_auc'],
            s=85,
            color=colors.get(row['selector_family'], '#777777'),
            marker=markers.get(row['dataset_name'], 'o'),
            alpha=0.9,
        )
        label = f"{dataset_label(row['dataset_name'])} {model_label(row['model'])} {method_label(row['selector'])}"
        ax.text(row['nogueira_stability'] + 0.004, row['oot_auc'] + 0.0007, label, fontsize=7)
    ax.set_xlabel('Nogueira stability')
    ax.set_ylabel('OOT AUC')
    ax.set_title(title)
    ax.grid(alpha=0.25)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # Lightweight custom legend.
    for family, color in colors.items():
        ax.scatter([], [], color=color, label=family_label(family), s=80)
    ax.legend(frameon=False, loc='lower right')
    save_fig(fig, plot_file)
    add_manifest(plot_file, title, [source_paths['cross_final']], 'created', '', caption)
except Exception as exc:
    add_manifest(plot_file, title, [source_paths['cross_final']], 'skipped', str(exc), caption)

manifest = pd.DataFrame(manifest_rows, columns=['plot_file', 'plot_title', 'data_source_files', 'status', 'reason_if_skipped', 'notes_for_report'])
manifest.to_csv(MANIFEST_PATH, index=False)

print('\nData availability audit:')
print(manifest[['plot_file', 'status', 'reason_if_skipped']].to_string(index=False))
print('\nCreated PNG files:')
for path in sorted(OUTPUT_DIR.glob('*.png')):
    print(' -', rel(path))
print('\nSkipped plots:')
skipped = manifest[manifest['status'] == 'skipped']
if skipped.empty:
    print(' - none')
else:
    for _, row in skipped.iterrows():
        print(f" - {row['plot_file']}: {row['reason_if_skipped']}")
print('\nManifest:', rel(MANIFEST_PATH))


## Data Availability Audit

The final code cell prints the audit table and writes the same information to `results/analysis_plots/plot_manifest.csv`. Skipped plots are intentional when required prediction-level or period-level score artifacts are unavailable.
